# Transformer Training on GSM8K

**Before running:** Make sure GPU is enabled — Runtime → Change runtime type → T4 GPU

Run cells in order.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/transformer-project/output'
print(f'Checkpoints will be saved to: {SAVE_DIR}')

In [ ]:
import os

REPO_DIR = '/content/transformer-project'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/KKD2005/transformer-project.git {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

!pip install -q -r requirements.txt

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import sys, os
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'data'))

TOKENIZER_PATH = os.path.join(REPO_DIR, 'data', 'gsm8k_tokenizer.json')
CORPUS_PATH    = os.path.join(REPO_DIR, 'data', 'tokenizer_corpus.txt')

from datasets import load_dataset
from generate_dataset import GSM8KDataset

print('Loading GSM8K dataset...')
raw_dataset = load_dataset('openai/gsm8k', 'main')
print(f'Train examples: {len(raw_dataset["train"])}')

tmp_ds = GSM8KDataset(raw_dataset, tokenizer=None)
tmp_ds.create_tokenizer_txt(CORPUS_PATH)
print(f'Corpus written to {CORPUS_PATH}')

from tokenizers import ByteLevelBPETokenizer
tok = ByteLevelBPETokenizer()
tok.train(
    files=[CORPUS_PATH],
    vocab_size=8192,
    min_frequency=2,
    special_tokens=['<pad>', '<eos>', '<bos>', '<problem>', '</problem>',
                    '<reasoning>', '</reasoning>', '<answer>', '</answer>']
)
tok.save(TOKENIZER_PATH)
print(f'Tokenizer saved to {TOKENIZER_PATH}')

In [ ]:
import os; os.environ['OUTPUT_DIR'] = SAVE_DIR

!python training/train.py